# Feature Selection & Engineering
1. Initial feature selection based on EDA and business understanding
2. Outlier treatment (risk-aware)
3. Feature scaling and normalisation
4. Creation of financial risk ratios
5. Categorical encoding


## 1. Initial feature selection based on EDA and business understanding
The objective is to retain variables that:

1. Are **available at application time**,
2. Have **economic meaning** in a credit risk context,
3. Show **directional relationship with default**, even if weak,
4. Avoid unnecessary noise, leakage, or instability given the rare-event setting.


### Scope and Exclusions

Only variables observable **before loan approval** are considered. As a result:

**Explicitly excluded variables include:**
* Loan status refinements after origination
* Payment history variables observed after the loan was issued
* Any outcome-driven or time-forward information

This preserves a realistic credit-scoring setup and avoids data leakage.


### Core Feature Groups

Feature selection is organised by **economic interpretation**, not just statistical correlation.


#### Borrower Financial Capacity

These variables describe the borrower’s ability to service debt.

**Retained features**
* `annual_income`
* `debt_to_income`
* `homeownership`

**Rationale**
* Income captures earning capacity but is skewed → requires transformation.
* DTI directly measures financial stress and shows weak but consistent directional signal.
* Homeownership differentiates fixed obligation structures (rent vs mortgage vs own).


#### Employment Stability

**Retained features**
* `emp_length`
* `verified_income`

**Rationale**
* Employment length shows heterogeneity but contributes stability context.
* Income verification adds reliability information beyond raw income.


#### Credit History Length and Structure

**Retained features**
* `earliest_credit_line`
* `total_credit_lines`
* `open_credit_lines`

**Rationale**
* Credit history length captures borrower maturity but does not imply lower risk on its own.
* Number of credit lines reflects exposure and complexity of obligations.
* These variables show moderate differences between default and non-default groups and complement leverage metrics.


#### Past Delinquency and Adverse Credit Events

These features are economically intuitive but **require careful selection due to sparsity**.

**Retained features**
* `months_since_last_delinq`
* `public_record_bankrupt`

**Explicitly excluded features**
* `num_historical_failed_to_pay`
* `delinq_2yr`
* `months_since_90d_late`

**Rationale**

* `num_historical_failed_to_pay` shows:
  * Heavy concentration at zero
  * Strong overlap between defaulted and non-defaulted borrowers
  * Unstable behaviour given only 7 default observations
    Its weak positive correlation with default reflects **directional intuition**, but not robust standalone signal.

* `delinq_2yr` exhibits:
  * Median equal to zero for both classes
  * Near-zero variance among defaulters
  * Counterintuitive mean behaviour (lower for defaulters), strongly suggesting **small-sample artifacts**

* `months_since_90d_late` shows:
  * Almost no variability in the default class
  * Collapse to a single value for defaulters

> In a rare-event setting, variables with sparse distributions and unstable class behaviour introduce noise and reduce generalisation performance.
> These features were therefore excluded in favour of more stable credit history proxies.


#### Legal and Tax Signals

**Retained features**
* `tax_liens`

**Rationale**
* Despite low frequency, tax liens show the **strongest categorical default rate differences**.
* Economically intuitive red-flag variable, retained with regularisation to control variance.


#### Loan Characteristics

**Retained features**
* `loan_amount`
* `loan_purpose`
* `application_type`

**Rationale**
* Loan purpose captures intent and risk profile (e.g. housing-related loans show higher default rates).
* Joint vs individual applications show minimal difference but are retained for completeness.
* Loan amount reflects exposure and interacts with income and leverage.


### Variables Excluded from Modeling

The following variables are excluded due to **low signal, instability, or redundancy**:
* Sparse delinquency counters with overlapping class distributions
* Variables with near-zero variance among defaulters
* Post-origination payment variables
* Highly granular categorical levels with insufficient observations
* Raw versions of variables replaced by transformed equivalents (e.g. unscaled income)

This helps reduce noise and overfitting in an already imbalanced dataset.


### Feature Transformation Strategy

Based on EDA findings:

* **Log-transform or cap**

  * `annual_income`
  * credit line counts
* **Scale robustly**

  * DTI and ratio-based features
* **Encode categoricals**

  * One-hot encoding with rare-category grouping
* **Preserve monotonicity where possible**

  * Useful for explainability in logistic or tree-based models

### 5. Modeling Philosophy

Feature selection is intentionally **conservative**:

* Preference is given to **economically interpretable and stable variables**
* Weak univariate signals are discarded when they are:
  * Sparse
  * Unstable
  * Likely driven by sampling noise

Given the **~1.5% default rate**, model performance will rely more on:

* Feature interactions
* Regularisation
* Threshold optimisation

rather than on any single dominant predictor.

In [8]:
import pandas as pd
import numpy as np

df = pd.read_csv('loans_full_schema.csv')
# df_credit = df[df["loan_status"].isin(["Charged Off", "Fully Paid"])]
df_credit = df.loc[
    df["loan_status"].isin(["Charged Off", "Fully Paid"])
].copy()


# Selected features based on EDA + credit intuition
num_features = [
    # Financial capacity
    "annual_income",
    "annual_income_joint",
    "debt_to_income",

    
    # Employment stability
    "emp_length",
    
    # Credit history & structure
    "earliest_credit_line",
    "total_credit_lines",
    "open_credit_lines",
    "total_debit_limit",
    "num_total_cc_accounts",
    "num_open_cc_accounts",

    # Past delinquency / adverse events
    "months_since_last_delinq",
    
    # Loan characteristics
    "loan_amount",
    
]

cat_features = [
    "homeownership",
    "verified_income",
    "public_record_bankrupt",
    "tax_liens",
    "loan_purpose",
    "application_type",
    "state",
    "emp_title"
]

features = num_features + cat_features

df_model = df_credit[features + ["loan_status"]].copy()
print(f"Modeling dataset shape: {df_model.shape}")
df_model.head()

Modeling dataset shape: (454, 21)


,annual_income,annual_income_joint,debt_to_income,emp_length,earliest_credit_line,total_credit_lines,open_credit_lines,total_debit_limit,num_total_cc_accounts,num_open_cc_accounts,...,loan_amount,homeownership,verified_income,public_record_bankrupt,tax_liens,loan_purpose,application_type,state,emp_title,loan_status
18,210000.0,NaN,9.53,10.0,2003,18,7,17500,6,2,...,5000,MORTGAGE,Verified,0,0,medical,individual,IL,operational risk manager,Fully Paid
19,83000.0,NaN,18.44,1.0,2005,11,6,8300,7,4,...,20000,MORTGAGE,Source Verified,0,0,debt_consolidation,individual,CA,welder,Fully Paid
34,140000.0,NaN,13.82,10.0,1993,21,12,58100,13,9,...,15000,MORTGAGE,Not Verified,0,0,debt_consolidation,individual,CA,deputy,Fully Paid
35,70000.0,NaN,0.00,1.0,2004,13,3,0,11,3,...,2400,OWN,Source Verified,0,0,small_business,individual,MD,armed protection officer,Fully Paid
107,44000.0,NaN,24.77,2.0,2011,16,9,9500,8,6,...,7200,MORTGAGE,Verified,0,0,home_improvement,individual,FL,crew chief,Fully Paid


## 2. Outlier treatment (risk-aware)

## 3. Feature scaling and normalisation


## 4. Creation of financial risk ratios

## 5. Categorical encoding